# Beyond - Agent Harness Engineering

This notebook turns reliability claims into deterministic drills: inspect a trajectory, crash and resume it, expose an unknown tool outcome, make context drift visible, break policy and tools, and compare two harnesses under identical faults.

1. Read the lesson page (`docs/beyond/harness.md`).
2. Open this notebook with `./notebook.sh harness`.
3. Answer the `Question:` / `Answer:` cells below.
4. When you're ready, ask a coding agent to grade your notebook.

Partial work is fine. Blank `Answer: ""` strings are skipped, not counted wrong. If you'd like a hint instead of a grade, write the request inline in the answer string and the agent will tutor first.

In [ ]:
import json
import shutil
from pathlib import Path

from g2c.agent.agent import Agent
from g2c.harness import (
    Budgets, Event, EventLog, HarnessAgent, Permission, ToolRunner,
    compact_context, estimate_tokens,
)
from g2c.harness.context import render_event
from g2c.inference import Backend, BackendInfo, InferenceResult
from g2c.tools import Tool, ToolCall, ToolError, ToolRegistry

WORK = Path("data/work/beyond-harness")
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir(parents=True)
print(f"workspace: {WORK}")

In [ ]:
class ScriptedBackend(Backend):
    """A deterministic backend; its script may inspect each prompt."""

    def __init__(self, script, crash_after=None):
        self.script = script
        self.crash_after = crash_after
        self.calls = 0
        self.attempts = 0
        self.prompts = []

    @property
    def info(self):
        return BackendInfo("fake", "scripted")

    def complete(self, prompt, *, max_new_tokens=128, temperature=1.0,
                 top_k=None, top_p=None):
        self.attempts += 1
        if self.crash_after is not None and self.calls >= self.crash_after:
            raise RuntimeError("injected backend crash")
        self.prompts.append(prompt)
        completion = (self.script(prompt, self.calls)
                      if callable(self.script) else self.script[self.calls])
        self.calls += 1
        return InferenceResult(prompt=prompt, completion=completion,
                               prompt_tokens=None, completion_tokens=None,
                               latency_ms=1.0, backend=self.info)


def sandbox_registry(sandbox: Path, counters: dict) -> ToolRegistry:
    notes = sandbox / "notes.txt"

    def append_line(text: str) -> str:
        counters["append_calls"] = counters.get("append_calls", 0) + 1
        with notes.open("a", encoding="utf-8") as f:
            f.write(text + "\n")
        return f"appended {text!r}"

    def flaky(text: str) -> str:
        counters["flaky_calls"] = counters.get("flaky_calls", 0) + 1
        if counters["flaky_calls"] <= counters.get("flaky_fail_first", 0):
            raise ToolError("connection timed out")
        return f"ok {text!r}"

    def missing(text: str) -> str:
        counters["missing_calls"] = counters.get("missing_calls", 0) + 1
        raise ToolError(f"file not found: {text}")

    schema = {"type": "object",
              "properties": {"text": {"type": "string"}},
              "required": ["text"]}
    return ToolRegistry([
        Tool("append_line", "Append one line", schema, append_line),
        Tool("flaky", "Fail transiently on cue", schema, flaky),
        Tool("missing", "Always report a missing file", schema, missing),
    ])


ACT = 'Action: append_line\nAction Input: {{"text": "{text}"}}'
FINAL = "Final Answer: done"

## Exercise 1 — Read a trajectory

Run a two-action task, then read the event log as a system record rather than a chat transcript.

In [ ]:
counters = {}
run_dir = WORK / "ex1"
run_dir.mkdir()
agent = HarnessAgent(
    ScriptedBackend([ACT.format(text="alpha"), ACT.format(text="beta"), FINAL]),
    sandbox_registry(run_dir, counters),
    EventLog(run_dir / "events.jsonl"),
)
result = agent.run("write alpha then beta to notes.txt")
print(f"stopped={result.stopped_reason!r}, notes={(run_dir / 'notes.txt').read_text()!r}")
for line in (run_dir / "events.jsonl").read_text().splitlines():
    event = json.loads(line)
    print(f"{event['step']:>2} {event['type']:<13} {event.get('call_id') or '':<8}",
          str(event['payload'])[:76])

In [ ]:
"Question: Point to two event-log records that a plain message transcript would not reliably preserve, and say what post-hoc question each lets you answer."
"Answer: "

## Exercise 2 — Crash, resume, and confront ambiguity

First resume after a backend crash that happened after a recorded tool result. Then construct the harder window: intent is logged, the side effect lands, and the process dies before result logging.

In [ ]:
counters = {}
drill = WORK / "ex2-resume"
drill.mkdir()
crashing = HarnessAgent(
    ScriptedBackend([ACT.format(text="alpha"), ACT.format(text="beta")], crash_after=1),
    sandbox_registry(drill, counters), EventLog(drill / "events.jsonl"),
)
try:
    crashing.run("write alpha then beta")
except RuntimeError as exc:
    print(f"first process died: {exc}")

revived = HarnessAgent(
    ScriptedBackend([ACT.format(text="beta"), FINAL]),
    sandbox_registry(drill, counters), EventLog(drill / "events.jsonl"),
)
resumed = revived.resume()
print(f"resumed={resumed.stopped_reason!r}, notes={(drill / 'notes.txt').read_text()!r}")

In [ ]:
ambiguous = WORK / "ex2-unknown"
ambiguous.mkdir()
counters = {}
log = EventLog(ambiguous / "events.jsonl")
log.append("task", {"content": "append alpha once"})
log.append("tool_call", {"tool": "append_line",
                              "arguments": {"text": "alpha"}}, call_id="call_0")
# Simulate a landed effect followed by a crash before tool_result was logged.
(ambiguous / "notes.txt").write_text("alpha\n", encoding="utf-8")
runner = ToolRunner(sandbox_registry(ambiguous, counters), log)
outcome = runner.execute(ToolCall("append_line", {"text": "alpha"}, "call_0"))
status = [e.payload["status"] for e in log.replay() if e.type == "tool_result"][-1]
print(f"status={status!r}, error={outcome.is_error}, tool reruns={counters.get('append_calls', 0)}")
print(f"notes remained exactly: {(ambiguous / 'notes.txt').read_text()!r}")

In [ ]:
"Question: Why can the later backend crash resume cleanly, while the unresolved call has an unknown outcome? Explain what call-id dedupe does and does not guarantee, and name one stronger mechanism that could close the gap."
"Answer: "

## Exercise 3 — Make context drift live

The backend emits four bulky filler actions. On its fifth turn it writes `TARGET` only if the original task is still visible; otherwise it writes `DRIFT`. Run that same backend through both context policies.

In [ ]:
def naive_context(events, budget):
    lines = [render_event(e) for e in events if render_event(e)]
    while lines and sum(estimate_tokens(line) for line in lines) > budget:
        lines.pop(0)
    return lines


def drift_script(prompt, call_index):
    if call_index < 4:
        return ACT.format(text=f"filler-{call_index}-" + "x" * 180)
    if call_index == 4:
        task_visible = ("Task: Keep TARGET" in prompt or
                        "Question: Keep TARGET" in prompt)
        return ACT.format(text="TARGET" if task_visible else "DRIFT")
    return FINAL


for policy_name, policy in [("naive", naive_context), ("extractive", compact_context)]:
    run_dir = WORK / f"ex3-{policy_name}"
    run_dir.mkdir()
    backend = ScriptedBackend(drift_script)
    result = HarnessAgent(
        backend, sandbox_registry(run_dir, {}), EventLog(run_dir / "events.jsonl"),
        budgets=Budgets(max_steps=7, context_tokens=180), context_policy=policy,
    ).run("Keep TARGET visible while processing filler")
    lines = (run_dir / "notes.txt").read_text().splitlines()
    task_visible = "Task: Keep TARGET" in backend.prompts[4]
    print(f"{policy_name:<10} task_visible_turn_5={task_visible!s:<5} last_line={lines[-1]!r} "
          f"stop={result.stopped_reason!r}")

In [ ]:
"Question: Compare the two live runs: how did losing the task change behavior? What narrow invariants does compact_context actually preserve? Where should resolved instructions such as applicable AGENTS.md rules live, and what richer state would a production system need to represent separately?"
"Answer: "

## Exercise 4 — Break tools and policy

Exercise all four branches: transient retry, deterministic failure, permission refusal, and repeat stopping. The permission check is application policy inside Python—not containment of the tool implementation.

In [ ]:
# Transient: one retry succeeds.
run_dir = WORK / "ex4-transient"; run_dir.mkdir()
counters = {"flaky_fail_first": 1}
result = HarnessAgent(
    ScriptedBackend(['Action: flaky\nAction Input: {"text": "x"}', FINAL]),
    sandbox_registry(run_dir, counters), EventLog(run_dir / "events.jsonl"),
).run("call flaky")
print("transient   ", result.stopped_reason, counters)

# Deterministic: the same missing path is attempted once, then shown to the model.
run_dir = WORK / "ex4-deterministic"; run_dir.mkdir()
counters = {}
result = HarnessAgent(
    ScriptedBackend(['Action: missing\nAction Input: {"text": "ghost.txt"}', FINAL]),
    sandbox_registry(run_dir, counters), EventLog(run_dir / "events.jsonl"),
).run("inspect ghost.txt")
print("deterministic", result.stopped_reason, counters)

# Denied: no side effect. This policy is still not an OS sandbox.
run_dir = WORK / "ex4-denied"; run_dir.mkdir()
counters = {}
result = HarnessAgent(
    ScriptedBackend([ACT.format(text="forbidden"), FINAL]),
    sandbox_registry(run_dir, counters), EventLog(run_dir / "events.jsonl"),
    permissions={"append_line": Permission.DENY},
).run("attempt a denied append")
print("denied      ", result.stopped_reason, "file_exists=", (run_dir / "notes.txt").exists())

# Repeat: stop before executing the duplicate when max_repeats=1.
run_dir = WORK / "ex4-repeat"; run_dir.mkdir()
counters = {}
result = HarnessAgent(
    ScriptedBackend([ACT.format(text="alpha")] * 5),
    sandbox_registry(run_dir, counters), EventLog(run_dir / "events.jsonl"),
    budgets=Budgets(max_steps=5, max_repeats=1),
).run("do not loop")
print("repeat      ", result.stopped_reason, counters)

In [ ]:
"Question: Explain why transient and deterministic failures take different paths, why a denied call is not retried, and why the permission table is not true sandboxing. What does the repeat budget guarantee—and what does it not solve?"
"Answer: "

## Exercise 5 — Controlled harness comparison

Required and model-free. Hold the scripted backend, task, tools, and fault schedule fixed; swap only Module 19's loop and `HarnessAgent`. Success uses scenario-specific state checks, never `Final Answer` alone.

In [ ]:
def run_matrix_case(harness_name, scenario):
    run_dir = WORK / f"matrix-{harness_name}-{scenario}"
    shutil.rmtree(run_dir, ignore_errors=True)
    run_dir.mkdir()
    counters = {"flaky_fail_first": 1}
    registry = sandbox_registry(run_dir, counters)
    resumed = False

    if scenario == "clean":
        backend = ScriptedBackend([ACT.format(text="alpha"), FINAL])
        task, permissions = "append alpha once", {}
    elif scenario == "transient":
        backend = ScriptedBackend(['Action: flaky\nAction Input: {"text": "x"}', FINAL])
        task, permissions = "call flaky until it succeeds", {}
    elif scenario == "crash":
        backend = ScriptedBackend([ACT.format(text="alpha")], crash_after=1)
        task, permissions = "append alpha once, then finish", {}
    elif scenario == "context":
        backend = ScriptedBackend(drift_script)
        task, permissions = "Keep TARGET visible while processing filler", {}
    elif scenario == "repeat":
        backend = ScriptedBackend([ACT.format(text="alpha")] * 5)
        task, permissions = "append alpha once; do not repeat", {}
    else:  # permission
        backend = ScriptedBackend([ACT.format(text="forbidden"), FINAL])
        task = "do not write without permission"
        permissions = {"append_line": Permission.DENY}

    backends = [backend]
    try:
        if harness_name == "module19":
            result = Agent(backend, registry, max_steps=7, plan=False,
                           scratchpad_max_chars=720).run(task)
            stop = result.stopped_reason
        else:
            agent = HarnessAgent(
                backend, registry, EventLog(run_dir / "events.jsonl"),
                permissions=permissions,
                budgets=Budgets(max_steps=7, max_repeats=1, context_tokens=180),
            )
            result = agent.run(task)
            stop = result.stopped_reason
    except RuntimeError:
        stop = "process_crash"
        if harness_name == "harness" and scenario == "crash":
            followup = ScriptedBackend([FINAL])
            backends.append(followup)
            result = HarnessAgent(
                followup, registry, EventLog(run_dir / "events.jsonl"),
                budgets=Budgets(max_steps=7, max_repeats=1, context_tokens=180),
            ).resume()
            resumed, stop = True, result.stopped_reason

    notes = ((run_dir / "notes.txt").read_text().splitlines()
             if (run_dir / "notes.txt").exists() else [])
    checks = {
        "clean": notes == ["alpha"],
        "transient": counters.get("flaky_calls", 0) == 2,
        "crash": notes == ["alpha"] and stop == "final_answer",
        "context": bool(notes) and notes[-1] == "TARGET",
        "repeat": notes == ["alpha"] and stop in {"duplicate_action", "repeat_budget"},
        "permission": notes == [],
    }
    expected_alpha = 1 if scenario in {"clean", "crash", "repeat"} else 0
    return {
        "harness": harness_name, "scenario": scenario, "success": checks[scenario],
        "model_calls": sum(b.attempts for b in backends),
        "tool_execs": sum(counters.get(k, 0) for k in ("append_calls", "flaky_calls", "missing_calls")),
        "resumes": int(resumed),
        "retries": max(0, counters.get("flaky_calls", 0) - 1),
        "duplicate_effects": max(0, notes.count("alpha") - expected_alpha),
        "stop": stop,
    }


scenarios = ["clean", "transient", "crash", "context", "repeat", "permission"]
rows = [run_matrix_case(h, s) for s in scenarios for h in ("module19", "harness")]
print(f"{'scenario':<11} {'harness':<9} {'ok':<5} {'model':>5} {'tools':>5} {'resume':>6} {'retry':>5} {'dupes':>5} stop")
for row in rows:
    print(f"{row['scenario']:<11} {row['harness']:<9} {str(row['success']):<5} "
          f"{row['model_calls']:>5} {row['tool_execs']:>5} {row['resumes']:>6} {row['retries']:>5} "
          f"{row['duplicate_effects']:>5} {row['stop']}")

In [ ]:
"Question: Interpret the controlled table. Which rows show parity, which differ, and what specific harness mechanism caused each difference? Use exact state and operational metrics—not Final Answer alone—and report null rows honestly."
"Answer: "

## Exercise 6 — Optional ProdLM transfer check

Run this cell only if you want a nondeterministic transfer check and have completed `./prodlm.sh`. Use one ProdLM first. Repeat a clean task and one faulted task with both harnesses, preserve the exact verifier, and distinguish model-format failures from harness-policy failures. A second ProdLM is stretch work.

A production-strength backend is intentional here: with a much weaker model, basic instruction-following failures can dominate so completely that the harness difference becomes uninformative.

In [ ]:
from g2c.inference import load_prodlm_backend, prodlm_manifest_exists

if not prodlm_manifest_exists():
    print("ProdLM is not configured. This optional exercise is skipped.")
else:
    prod_backend = load_prodlm_backend()
    task = ("Use append_line exactly once with text alpha. Do not write any other "
            "text. Finish only after the tool succeeds.")
    for harness_name in ("module19", "harness"):
        run_dir = WORK / f"prodlm-{harness_name}"
        shutil.rmtree(run_dir, ignore_errors=True)
        run_dir.mkdir()
        registry = sandbox_registry(run_dir, {})
        if harness_name == "module19":
            result = Agent(prod_backend, registry, max_steps=5, plan=False,
                           temperature=0.0).run(task)
            stop = result.stopped_reason
        else:
            result = HarnessAgent(
                prod_backend, registry, EventLog(run_dir / "events.jsonl"),
                budgets=Budgets(max_steps=5), temperature=0.0,
            ).run(task)
            stop = result.stopped_reason
        notes = ((run_dir / "notes.txt").read_text().splitlines()
                 if (run_dir / "notes.txt").exists() else [])
        print(f"{harness_name:<9} exact_success={notes == ['alpha']} "
              f"notes={notes!r} stop={stop!r}")
    print("For the faulted transfer row, repeat with flaky_fail_first=1 and "
          "record model-format failures separately from retry-policy failures.")

In [ ]:
"Question (optional): Did the ProdLM subset reproduce the scripted comparison? Separate model-format or tool-selection failures from harness-policy failures, and explain why a weak backend could hide the harness effect."
"Answer: "

When complete, ask a coding agent to grade your notebook. Partial work is fine: the agent should grade answered questions and implemented sections, then skip blank prompts.